# Fase 4 — Modeling | CardioRisk · CRISP-DM
Entrenamiento y optimización. Métrica principal: **Recall** (minimizar falsos negativos).
Carga artefactos de F3 vía %store — fallback a pipeline completo si no disponible.

In [ ]:
# BLOQUE 1 — INSTALACIONES + CARGA DE ARTEFACTOS
!pip install -q kagglehub imbalanced-learn xgboost shap

import warnings, joblib
import numpy as np
import pandas as pd
import matplotlib, matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import recall_score, precision_score, f1_score, roc_auc_score, accuracy_score, make_scorer
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
warnings.filterwarnings('ignore')

matplotlib.rcParams.update({
    'figure.facecolor': '#0a0f1a', 'axes.facecolor': '#0d1526',
    'axes.edgecolor': '#1a2c3d',   'axes.labelcolor': '#e2e8f0',
    'text.color': '#e2e8f0',       'xtick.color': '#7a8fa8',
    'ytick.color': '#7a8fa8',      'grid.color': '#1a2c3d',
    'savefig.facecolor': '#0a0f1a'
})

# Carga de artefactos desde F3
try:
    %store -r X_train_sm
    %store -r y_train_sm
    %store -r X_val_sc
    %store -r y_val
    %store -r X_test_sc
    %store -r y_test
    %store -r scaler
    %store -r FEATURES_FINALES
    print(f'✓ Artefactos cargados desde F3 — {len(FEATURES_FINALES)} features')
except:
    print('⚠ %store no disponible — ejecutando CANON completo')
    import kagglehub, os
    try:
        path = kagglehub.dataset_download('jocelyndumlao/cardiovascular-disease-dataset')
        csv_path = next(os.path.join(r,f) for r,_,fs in os.walk(path) for f in fs if f.endswith('.csv'))
        df = pd.read_csv(csv_path)
    except:
        df = pd.read_csv('/content/cardiovascular_disease_dataset.csv')
    df.columns = df.columns.str.lower().str.strip()
    df = df.dropna()
    df['slope_x_oldpeak'] = df['slope'] * df['oldpeak']
    df['log_oldpeak']     = np.log1p(df['oldpeak'])
    CATEGORICAL = ['gender', 'chestpain', 'restingrelectro']
    TARGET = 'target'
    df_enc = pd.get_dummies(df, columns=CATEGORICAL, drop_first=False)
    EXCLUIR = [TARGET, 'patientid']
    FEATURES_FINALES = [c for c in df_enc.columns if c not in EXCLUIR]
    X = df_enc[FEATURES_FINALES]; y = df_enc[TARGET]
    X_temp,X_test,y_temp,y_test = train_test_split(X,y,test_size=0.15,random_state=42,stratify=y)
    X_train,X_val,y_train,y_val = train_test_split(X_temp,y_temp,test_size=0.1765,random_state=42,stratify=y_temp)
    scaler = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)
    X_val_sc   = scaler.transform(X_val)
    X_test_sc  = scaler.transform(X_test)
    sm = SMOTE(random_state=42)
    X_train_sm, y_train_sm = sm.fit_resample(X_train_sc, y_train)
    print(f'✓ CANON ejecutado — {len(FEATURES_FINALES)} features')


In [ ]:
# BLOQUE 2 — ENTRENAMIENTO DE 5 MODELOS EN VALIDACIÓN
models = {
    'LogisticRegression': LogisticRegression(random_state=42, max_iter=1000),
    'RandomForest':       RandomForestClassifier(random_state=42, n_estimators=100),
    'GradientBoosting':   GradientBoostingClassifier(random_state=42),
    'XGBoost':            XGBClassifier(random_state=42, eval_metric='logloss', verbosity=0),
    'SVM':                SVC(random_state=42, probability=True)
}

results = []
for name, model in models.items():
    model.fit(X_train_sm, y_train_sm)
    y_pred = model.predict(X_val_sc)
    y_prob = model.predict_proba(X_val_sc)[:, 1]
    results.append({
        'Modelo': name,
        'Recall':    recall_score(y_val, y_pred),
        'Precision': precision_score(y_val, y_pred),
        'F1':        f1_score(y_val, y_pred),
        'AUC-ROC':   roc_auc_score(y_val, y_prob),
        'Accuracy':  accuracy_score(y_val, y_pred)
    })

results_df = pd.DataFrame(results).sort_values('Recall', ascending=False)
print('=== Resultados en Validación (ordenados por Recall) ===')
print(results_df.round(4).to_string(index=False))


In [ ]:
# BLOQUE 3 — VISUALIZACIÓN COMPARATIVA
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
metrics_kpi = [('Recall', 0.80), ('AUC-ROC', 0.85), ('F1', 0.80)]
colors_ok = '#00ff88'; colors_fail = '#ef4444'

for ax, (metric, kpi) in zip(axes, metrics_kpi):
    vals = results_df[metric].values
    names = results_df['Modelo'].values
    colors = [colors_ok if v >= kpi else colors_fail for v in vals]
    bars = ax.bar(names, vals, color=colors, alpha=0.8, edgecolor='#1a2c3d')
    ax.axhline(y=kpi, color='#fbbf24', linestyle='--', lw=2, label=f'KPI={kpi}')
    ax.set_title(metric, fontsize=12, fontweight='bold')
    ax.set_ylim(0, 1.1)
    ax.tick_params(axis='x', rotation=45)
    ax.legend(facecolor='#0d1526', edgecolor='#1a2c3d', labelcolor='#e2e8f0', fontsize=9)
    ax.grid(axis='y', alpha=0.3)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01, f'{val:.3f}', ha='center', fontsize=8)

plt.suptitle('Comparación de Modelos — Validación', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/f4_model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# BLOQUE 4 — OPTIMIZACIÓN DEL MEJOR MODELO (mayor Recall en val)
best_name = results_df.iloc[0]['Modelo']
print(f'Mejor modelo por Recall: {best_name}')

param_grids = {
    'LogisticRegression': {'C': [0.01, 0.1, 1, 10], 'solver': ['lbfgs', 'liblinear']},
    'RandomForest':       {'n_estimators': [100, 200], 'max_depth': [5, 10, None], 'min_samples_split': [2, 5]},
    'GradientBoosting':   {'n_estimators': [100, 200], 'max_depth': [3, 5], 'learning_rate': [0.05, 0.1]},
    'XGBoost':            {'n_estimators': [100, 200], 'max_depth': [3, 5], 'learning_rate': [0.05, 0.1]},
    'SVM':                {'C': [0.1, 1, 10], 'kernel': ['rbf', 'linear']}
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
recall_scorer = make_scorer(recall_score)

grid_search = GridSearchCV(
    models[best_name], param_grids[best_name],
    cv=cv, scoring=recall_scorer, n_jobs=-1, verbose=0
)
grid_search.fit(X_train_sm, y_train_sm)

best_model = grid_search.best_estimator_
y_pred_opt = best_model.predict(X_val_sc)
y_prob_opt = best_model.predict_proba(X_val_sc)[:, 1]
recall_opt = recall_score(y_val, y_pred_opt)
auc_opt    = roc_auc_score(y_val, y_prob_opt)

print(f'Mejores hiperparámetros: {grid_search.best_params_}')
print(f'Recall val optimizado  : {recall_opt:.4f}')
print(f'AUC-ROC val optimizado : {auc_opt:.4f}')


In [ ]:
# BLOQUE 5 — CURVA ROC + GUARDAR MODELO
from sklearn.metrics import roc_curve

fpr, tpr, _ = roc_curve(y_val, y_prob_opt)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(fpr, tpr, color='#00ff88', lw=2, label=f'{best_name} (AUC={auc_opt:.3f})')
ax.plot([0,1],[0,1], color='#7a8fa8', lw=1, linestyle='--')
ax.fill_between(fpr, tpr, alpha=0.15, color='#00ff88')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate (Recall)')
ax.set_title(f'Curva ROC — {best_name}', fontsize=13, fontweight='bold')
ax.legend(loc='lower right', facecolor='#0d1526', edgecolor='#1a2c3d', labelcolor='#e2e8f0')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('/content/f4_roc_curve.png', dpi=300, bbox_inches='tight')
plt.show()

joblib.dump(best_model,      '/content/best_model.pkl')
joblib.dump(scaler,          '/content/scaler.pkl')
joblib.dump(FEATURES_FINALES,'/content/feature_cols.pkl')

print(f'→ F4 COMPLETO — Modelo: {best_name} | Recall val: {recall_opt:.3f} | AUC val: {auc_opt:.3f}')
